<a href="https://colab.research.google.com/github/mehrerm/TFM/blob/main/notebooks/carga_depuracion_densidadRT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



En esta aplicación implementaremos un modelo de puntuación de riesgo poblacional a la mortalidad debido a los cánceres más comunes y su relación con las cantidades y tecnologías usadas en radioterapia por país.

In [1]:
#Cargo o importo pandas, numpy, Matplotlib,
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
import statsmodels.api as sm
import requests
import unicodedata
import os
from pathlib import Path
from optbinning import OptimalBinning, Scorecard, BinningProcess
from urllib.parse import quote




#lugar donde se van guardando las figuras

output_path = Path("TFM/data/processed")
output_path.mkdir(parents=True, exist_ok=True)

fig_dir = "figuras"
os.makedirs(fig_dir, exist_ok=True)

#########################

# Carga, Exploración y Preparación de los datos sobre equipos de radioterapia y pacientes
Se utilizará una base de datos de instalaciones de equipos para radioterapia registrados DIRAC de la IAEA junto con los de la OMS, son datos reales, se buscará el año más actual posible con una cantidad de datos suficientes para hacer el estudio y que tenga lo menos posible la influencia del COVID-19, así como un año anterior a éste.


  donde aparecen reflejados los datos de los equipos de radioterapia y el año de sus ultimas actualizaciones a nivel the hardware, por otro lado, se descargaron los datos de la OMS, en el Global Cancer Observatory, donde se descargaron tanto las incidencias como las mortalidades por cáncer y sexo a nivel mundial https://gco.iarc.fr/overtime/en/dataviz/trends?populations=752&sexes=1_2&types=1&multiple_populations=1.
Asimismo, se descargó la información demográfica encontrada en la ONU en el apartado "United Nations World Population Prospects (WPP) para poder ver la densidad poblacional.
Por otro lado, también se ha visto que es necesario para entender mejor la calidad de vida y sobreviviencia, que el PIB per capita puede jugar un papel importante, por lo tanto, también se sumará ese valor a los paises a estudiar, en ese caso se usará una url directa que da los datos sin necesidad de guardarlos en el repositorio que se tiene para tal fin.

## Carga de datos

In [2]:
# Cargamos los datos

# URL base del repositorio
url_base = "https://raw.githubusercontent.com/mehrerm/TFM/main/data/raw/"

#DIRAC
#Datos de los equipos y centros de radioterapia por país

#url_dirac = (
 #   url_base + "DIRAC_Countries.xlsx"
#)

#WHO datos de densidad de equipos de RT por años, se carga a su vez los equipos
#de diagnostico

url_dirac = url_base + "RT_densidad_anio.csv"
dirac = pd.read_csv(url_dirac)
print(["Para los equipos de RT se tiene:"])
print(dirac.head(5))

#para los equipos de radiodiagnotico
url_diagnostico = url_base + "Diagnostico_densidad.csv"
diagnostico = pd.read_csv(url_diagnostico)
print(["Para los equipos de diagnostico se tiene:"])
print(diagnostico.head(5))



#GLOBOCAN
#datos de tipos de cancer más tratados con radioterapia, incidencia, mortalidad
#por año
# Archivos
files_cancer = {
    "lung": "dataset-asr-inc-and-mort-males-and-females-lung.csv",
    "breast": "dataset-asr-inc-and-mort-males-and-females-breast.csv",
    "prostate": "dataset-asr-inc-and-mort-males-and-females-prostat.csv",
    #"colon": "dataset-asr-inc-and-mort-males-and-females-colon.csv",
    "cervix": "dataset-asr-inc-and-mort-males-and-females-cervix-uterino.csv",
}

# Carga de datasets
dfd_cancer = []
print(["Para GLOBOCAN se tiene:"])
for cancer, filename in files_cancer.items():
    url = url_base + quote(filename)
    df_cancer = pd.read_csv(url)



    dfd_cancer.append(df_cancer)

    print(f"{cancer.upper():10s} -> shape: {df_cancer.shape}")

df_all_cancer = pd.concat(dfd_cancer, ignore_index=True)


#ONU
#Población mundial acorde con los datos de la ONU

url_pop = (
       url_base + "WPP2024_TotalPopulationBySex.csv.gz"
)


pop = pd.read_csv(url_pop, compression='gzip')
print(["Para la ONU se tiene:"])
print(pop.head(5))


# Banco mundial para estudiar los pib per capita
url_BM = (
    "https://api.worldbank.org/v2/country/all/indicator/NY.GDP.PCAP.CD"
    "?format=json&per_page=20000"
)



PIB_ = requests.get(url_BM).json()

df_PIB_raw = pd.DataFrame(PIB_[1])

df_PIB = df_PIB_raw[["countryiso3code", "country", "date", "value", "indicator"]].copy()
print(["Para el banco mundial se tiene"])
print(df_PIB.head(5))




['Para los equipos de RT se tiene:']
  IndicatorCode                                          Indicator ValueType  \
0     DEVICES22  Total density per million population: Radiothe...      text   
1     DEVICES22  Total density per million population: Radiothe...      text   
2     DEVICES22  Total density per million population: Radiothe...      text   
3     DEVICES22  Total density per million population: Radiothe...      text   
4     DEVICES22  Total density per million population: Radiothe...      text   

  ParentLocationCode   ParentLocation Location type SpatialDimValueCode  \
0                AFR           Africa       Country                 BFA   
1                AFR           Africa       Country                 COM   
2                WPR  Western Pacific       Country                 FSM   
3                AFR           Africa       Country                 ETH   
4                AFR           Africa       Country                 UGA   

                           Loca

/tmp/ipython-input-2092067397.py:65: DtypeWarning: Columns (2,3,4,7) have mixed types. Specify dtype option on import or set low_memory=False.
  pop = pd.read_csv(url_pop, compression='gzip')


['Para la ONU se tiene:']
   SortOrder  LocID Notes ISO3_code ISO2_code  SDMX_code  LocTypeID  \
0        NaN   5507   NaN       NaN       NaN        NaN        NaN   
1        NaN   5507   NaN       NaN       NaN        NaN        NaN   
2        NaN   5507   NaN       NaN       NaN        NaN        NaN   
3        NaN   5507   NaN       NaN       NaN        NaN        NaN   
4        NaN   5507   NaN       NaN       NaN        NaN        NaN   

  LocTypeName  ParentID                           Location  VarID Variant  \
0         NaN       NaN  ADB region: Central and West Asia      2  Medium   
1         NaN       NaN  ADB region: Central and West Asia      2  Medium   
2         NaN       NaN  ADB region: Central and West Asia      2  Medium   
3         NaN       NaN  ADB region: Central and West Asia      2  Medium   
4         NaN       NaN  ADB region: Central and West Asia      2  Medium   

   Time  MidPeriod    PopMale  PopFemale   PopTotal  PopDensity  
0  1950     1950.5

Se buscan Nan

In [3]:
df_all_cancer.isna().sum()

,0
Cancer id,0
Cancer label,0
Population id,0
Country label,0
Sex,0
Type,0
Year,0
ASR (World),0
Crude rate,0
Cumulative risk,0


## Descripción inicial de los datos

## Depuración y exploración de los datos

### ONU
Con esta dataset, se pretende conseguir la población global por año

In [4]:
print(pop.head())
print(pop["LocTypeName"].unique())

   SortOrder  LocID Notes ISO3_code ISO2_code  SDMX_code  LocTypeID  \
0        NaN   5507   NaN       NaN       NaN        NaN        NaN   
1        NaN   5507   NaN       NaN       NaN        NaN        NaN   
2        NaN   5507   NaN       NaN       NaN        NaN        NaN   
3        NaN   5507   NaN       NaN       NaN        NaN        NaN   
4        NaN   5507   NaN       NaN       NaN        NaN        NaN   

  LocTypeName  ParentID                           Location  VarID Variant  \
0         NaN       NaN  ADB region: Central and West Asia      2  Medium   
1         NaN       NaN  ADB region: Central and West Asia      2  Medium   
2         NaN       NaN  ADB region: Central and West Asia      2  Medium   
3         NaN       NaN  ADB region: Central and West Asia      2  Medium   
4         NaN       NaN  ADB region: Central and West Asia      2  Medium   

   Time  MidPeriod    PopMale  PopFemale   PopTotal  PopDensity  
0  1950     1950.5  35880.164  33333.260  69

In [5]:
# En LocTypeName, filtrar: países, escenario Medium
ONU_c = pop[
    (pop["LocTypeName"] == "Country/Area") &
    (pop["Variant"] == "Medium")
    #El escenario es Medium porque es el escenario más estandarizado
    #esto se debe a que también presenta datos extrapolados sergún distintos
    #criterios
   # (pop["Time"].isin([2016, 2022]))

].copy()

# Seleccionar y renombrar columnas
ONU_c = (
    ONU_c[["Location", "Time", "PopTotal", "PopMale","PopFemale", "ISO3_code", "LocID"]]
    .rename(columns={
    "Location": "Country_harmonized",
    "Time": "Year",
    "PopTotal": "Population",
    "PopMale": "Population_male",
    "PopFemale": "Population_female",
    "ISO3_code": "ISO3",
    "LocID": "Population id",
})
)

# Convertir tipos
ONU_c["Year"] = ONU_c["Year"].astype(int)

# PopTotal está en miles, pasar a personas
cols = ["Population", "Population_male", "Population_female"]
ONU_c[cols] = ONU_c[cols] * 1000

# Comprobaciones rápidas
print(ONU_c.shape)
print(ONU_c.head(5))

# Guardar CSV reducido
ONU_c.to_csv(
    "population_UN_WPP2024.csv",
    index=False
)

print(ONU_c["Country_harmonized"].unique())
ONU_c["Year"].unique()

(35787, 7)
       Country_harmonized  Year  Population  Population_male  \
336230            Burundi  1950   2254938.0        1080184.0   
336231            Burundi  1951   2305746.0        1105816.0   
336232            Burundi  1952   2355804.0        1130995.0   
336233            Burundi  1953   2405186.0        1155833.0   
336234            Burundi  1954   2454586.0        1180690.0   

        Population_female ISO3  Population id  
336230          1174755.0  BDI            108  
336231          1199930.0  BDI            108  
336232          1224809.0  BDI            108  
336233          1249353.0  BDI            108  
336234          1273896.0  BDI            108  
['Burundi' 'Comoros' 'Djibouti' 'Eritrea' 'Ethiopia' 'Kenya' 'Madagascar'
 'Malawi' 'Mauritius' 'Mayotte' 'Mozambique' 'Réunion' 'Rwanda'
 'Seychelles' 'Somalia' 'South Sudan' 'Uganda'
 'United Republic of Tanzania' 'Zambia' 'Zimbabwe' 'Angola' 'Cameroon'
 'Central African Republic' 'Chad' 'Congo'
 'Democratic Repu

array([1950, 1951, 1952, 1953, 1954, 1955, 1956, 1957, 1958, 1959, 1960,
       1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971,
       1972, 1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982,
       1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993,
       1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004,
       2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015,
       2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026,
       2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034, 2035, 2036, 2037,
       2038, 2039, 2040, 2041, 2042, 2043, 2044, 2045, 2046, 2047, 2048,
       2049, 2050, 2051, 2052, 2053, 2054, 2055, 2056, 2057, 2058, 2059,
       2060, 2061, 2062, 2063, 2064, 2065, 2066, 2067, 2068, 2069, 2070,
       2071, 2072, 2073, 2074, 2075, 2076, 2077, 2078, 2079, 2080, 2081,
       2082, 2083, 2084, 2085, 2086, 2087, 2088, 2089, 2090, 2091, 2092,
       2093, 2094, 2095, 2096, 2097, 2098, 2099, 21

In [6]:
ONU_c.isna().sum()

,0
Country_harmonized,0
Year,0
Population,0
Population_male,0
Population_female,0
ISO3,0
Population id,0


In [7]:
#nombres de paises, se estandarizarán los nombres de los paises de todos los
#dataset

country_mapping = {

    "Macau, China": "Macao",
    "Taiwan, China": "Taiwan",
    'China, Hong Kong SAR': "Hong Kong",
    "China, Macao SAR": "Macao",
    'China, Taiwan Province of China': "Taiwan",
    'China, Republic of China': "China",
    "Macao SAR, China": "Macao",

    'Lao PDR': "Lao",
    "Lao People's Democratic Republic": "Lao",



    'Czech Republic': 'Czechia',

    "Slovak Republic": "Slovakia",

    "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",

    "UK, England": "United Kingdom",
    "UK, Wales": "United Kingdom",
    "UK, Scotland": "United Kingdom",
    "UK, Northern Ireland": "United Kingdom",
    "UK, England and wales": "United Kingdom",


    "France (metropolitan)": "France",
    "France, Martinique": "Martinique",

    "USA": "United States of America",
    "United States" : "United States of America",
    "United States of America (the)": "United States of America",

    "Puerto Rico (US)": "Puerto Rico",



    "Korea, Republic of": "South Korea",
    "Republic of Korea" : "South Korea",
    "Korea, Rep.": "South Korea",
    "Korea, Democratic People's Republic of": "North Korea",


    "Iran, Islamic Republic of": "Iran",
    "Iran (Islamic Republic of)": "Iran",

    "Viet Nam": "Vietnam",

    'Venezuela, Bolivarian Republic of': 'Venezuela',
    'Venezuela (Bolivarian Republic of)': 'Venezuela',
    "Venezuela, RB": "Venezuela",

    'Bolivia, Plurinational State of': 'Bolivia',
    'Bolivia (Plurinational State of)': 'Bolivia',

    "Netherlands, Kingdom of the": "The Netherlands",

    "Republic of Moldova": "Moldova", # Comma added here

    "Kyrgyz Republic": "Kyrgyzstan",

    "Syrian Arab Republic": "Syria"

}

En la celda anterior presenta una lista de países cuyos nombres se han cambiado para que coincidan los 4 datasets.

In [8]:
# se limpian los caracteres de los paises y se cambian los nombres de aquellos
#que sean necesarios


ONU_c["Country_harmonized"] = ONU_c["Country_harmonized"].replace(country_mapping)

ONU_c["Country_harmonized"] = ONU_c["Country_harmonized"].apply(
    lambda x: unicodedata.normalize("NFKD", x)
        .encode("ASCII", "ignore")
        .decode("utf-8") if pd.notna(x) else x
)

print(ONU_c['Country_harmonized'].unique())
print(ONU_c.info())
print(ONU_c.isna().sum().sum())
print(ONU_c['Year'].unique())


['Burundi' 'Comoros' 'Djibouti' 'Eritrea' 'Ethiopia' 'Kenya' 'Madagascar'
 'Malawi' 'Mauritius' 'Mayotte' 'Mozambique' 'Reunion' 'Rwanda'
 'Seychelles' 'Somalia' 'South Sudan' 'Uganda'
 'United Republic of Tanzania' 'Zambia' 'Zimbabwe' 'Angola' 'Cameroon'
 'Central African Republic' 'Chad' 'Congo'
 'Democratic Republic of the Congo' 'Equatorial Guinea' 'Gabon'
 'Sao Tome and Principe' 'Algeria' 'Egypt' 'Libya' 'Morocco' 'Sudan'
 'Tunisia' 'Western Sahara' 'Botswana' 'Eswatini' 'Lesotho' 'Namibia'
 'South Africa' 'Benin' 'Burkina Faso' 'Cabo Verde' "Cote d'Ivoire"
 'Gambia' 'Ghana' 'Guinea' 'Guinea-Bissau' 'Liberia' 'Mali' 'Mauritania'
 'Niger' 'Nigeria' 'Saint Helena' 'Senegal' 'Sierra Leone' 'Togo'
 'Kazakhstan' 'Kyrgyzstan' 'Tajikistan' 'Turkmenistan' 'Uzbekistan'
 'China' 'Hong Kong' 'Macao' 'Taiwan' "Dem. People's Republic of Korea"
 'Japan' 'Mongolia' 'South Korea' 'Afghanistan' 'Bangladesh' 'Bhutan'
 'India' 'Iran' 'Maldives' 'Nepal' 'Pakistan' 'Sri Lanka'
 'Brunei Darussalam' 'C

Variables a utilizar:

* Country_harmonized: los países con información

* Year: año

* Population: población tanto hombres como mujeres

* Population_male: pobración de hombres

* Population_female: población de mujeres

Los datos de población utilizados en este estudio proceden de la base World Population Prospects 2024 elaborada por la División de Población del Departamento de Asuntos Económicos y Sociales de las Naciones Unidas (ONU). Según la metodología oficial de esta fuente, las estimaciones de población se basan en datos observados provenientes de censos nacionales, registros vitales y encuestas demográficas, mientras que las proyecciones de población comienzan a partir del año 2024.

Aunque el conjunto de datos incluye estimaciones hasta el año 2023, la disponibilidad y calidad de los datos demográficos recientes varía considerablemente entre países. En muchos casos, los últimos censos o registros vitales utilizados como base empírica corresponden a años anteriores, especialmente en el periodo posterior a la pandemia de COVID-19. Como consecuencia, los valores más recientes incorporan un mayor grado de interpolación y ajuste modelizado.

Por este motivo, y con el objetivo de evitar el uso de datos proyectados o altamente modelizados, y debido a las exploraciones posteriores con el resto de nuestras fuentes de datos, en este trabajo se selecciona el año 2022 como el año más reciente con una cobertura amplia y consistente de datos poblacionales basados mayoritariamente en información observada (y no extrapolada).

La utilización del año 2022 resulta, además, especialmente adecuada para su integración con los datos de mortalidad por cáncer de GLOBOCAN y los datos de infraestructura de radioterapia del registro DIRAC, que presentan una mayor disponibilidad y estabilidad en torno a ese periodo temporal. De este modo, la fusión de las fuentes se realiza minimizando sesgos del uso de proyecciones demográficas.

Asimismo, se utilizará el año 2016, ya que es un año también bastante consistente donde además, en GLOBOCAN, presenta buena recolección de datos sobre incidencias, algo que más adelante se demostrará que años posteriores, este valor estará ausente a la mayoría de países.


#WHO

En este base de datos tenemos todo lo referente a las unidades de radioterapia, las cuales son los aceleradores lineales y los equipos emisores gamma de cobalto. Su densidad segun población en el momento del registro, a lo largo de distintos años: '2021', '2017-2021', '2014', '2013', '2010'. Esto nos puede ayudar a entender si efectivamente el indice de mortalidad disminuye si aumentamos la densidad de estos equipos a lo largo del tiempo, este dato es importante ya que para ver efectivamente los efectos de un tratamiento, hay que ver al menos 5 años tras el registro del mismo. Por lo tanto, se buscará la correlación de equipos en un determinado año (2010, por ejemplo) y la mortalidad durante los 5 años posteriores (2015).

In [9]:
dirac.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 481 entries, 0 to 480
Data columns (total 34 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   IndicatorCode               481 non-null    object 
 1   Indicator                   481 non-null    object 
 2   ValueType                   481 non-null    object 
 3   ParentLocationCode          481 non-null    object 
 4   ParentLocation              481 non-null    object 
 5   Location type               481 non-null    object 
 6   SpatialDimValueCode         481 non-null    object 
 7   Location                    481 non-null    object 
 8   Period type                 481 non-null    object 
 9   Period                      481 non-null    object 
 10  IsLatestYear                481 non-null    bool   
 11  Dim1 type                   0 non-null      float64
 12  Dim1                        0 non-null      float64
 13  Dim1ValueCode               0 non-n

In [10]:
dirac["Period"].unique()

array(['2021', '2017-2021', '2014', '2013', '2010'], dtype=object)

In [11]:
dirac['Period'].value_counts()


,count
Period,
2010,172
2013,163
2017-2021,133
2021,10
2014,3


In [12]:
# List the columns you want to drop
columns_to_drop = [
    'IndicatorCode',
    'ValueType',
    'ParentLocationCode',
    'Location type',
    #'SpatialDimValueCode',
    'Period type',
    'IsLatestYear',
    'Dim1 type',
    'Dim1',
    'Dim1ValueCode',
    'Dim2 type',
    'Dim2',
    'Dim2ValueCode',
    'Dim3 type',
    'Dim3',
    'Dim3ValueCode',
    'DataSourceDimValueCode',
    'DataSource',
    'FactValueNumeric',
    'FactValueNumericPrefix',
    'FactValueUoM',
    'FactValueNumericLowPrefix',
    'FactValueNumericLow',
    'FactValueNumericHighPrefix',
    'FactValueNumericHigh',
    'FactValueTranslationID',
    "Indicator",
    'Language',
    'DateModified',
    'FactComments'
]

# Drop the columns from the DataFrame (e.g., dirac)
# I will use 'dirac' as it was the last dataframe being processed before the current context.
# If you want to drop columns from a different dataframe, replace 'dirac' with its name.
# The 'errors="ignore"' argument prevents an error if a column in the list doesn't exist.
dirac = dirac.drop(columns=columns_to_drop, errors="ignore")

# Display the columns after dropping to verify
print(dirac.head())


    ParentLocation SpatialDimValueCode                          Location  \
0           Africa                 BFA                      Burkina Faso   
1           Africa                 COM                           Comoros   
2  Western Pacific                 FSM  Micronesia (Federated States of)   
3           Africa                 ETH                          Ethiopia   
4           Africa                 UGA                            Uganda   

  Period  Value  
0   2021   0.00  
1   2021   0.00  
2   2021   0.00  
3   2021   0.09  
4   2021   0.09  


In [13]:
dirac_melted = dirac.copy()

# Rename columns to desired names
dirac = dirac.rename(columns={
    "Location": "Country_harmonized",
    "ParentLocation": "Region Name",
    "Total density per million population: Radiotherapy units": "RTUnit",
    "Period" : "Year_Units",
    "Value" : "RT_m",
    "SpatialDimValueCode" : "ISO3"
})





El siguiente paso con respecto a los años, tiene que ver con el historial que se tiene sobre los efectos de los tratamientos de cancer, por eso se suman 5 años.

No será lo mismo con los de diagnostico.

In [14]:
dirac["Country_harmonized"] = dirac["Country_harmonized"].replace(country_mapping)

dirac["Country_harmonized"] = dirac["Country_harmonized"].apply(
    lambda x: unicodedata.normalize("NFKD", x)
        .encode("ASCII", "ignore")
        .decode("utf-8") if pd.notna(x) else x
)

# Replace '2017-2021' with '2017' in Year_Units column
dirac['Year_Units'] = dirac['Year_Units'].replace('2017-2021', '2017')
# Convert 'Year_Units' to int
dirac['Year_Units'] = dirac['Year_Units'].astype(int)

# Filter out unwanted Year_Units (original periods 2021 and 2014) and explicitly create a copy
dirac = dirac[~dirac['Year_Units'].isin([2021, 2014])].copy()

# Calculate 'Year' by adding 5 years to 'Year_Units'
dirac["Year"] = dirac["Year_Units"] + 5

print(dirac['Country_harmonized'].unique())
print(dirac.info())
print(dirac.isna().sum().sum())
print(dirac['Year'].unique())

['Mozambique' 'Yemen' 'Cameroon' 'Nigeria' 'Mali' 'Madagascar'
 "Cote d'Ivoire" 'Angola' 'Rwanda' 'United Republic of Tanzania'
 'Cambodia' 'Tajikistan' 'Bangladesh' 'Ghana' 'Lao' 'Sudan' 'Uzbekistan'
 'Zambia' 'Indonesia' 'Kenya' 'Pakistan' 'Nepal' 'Kyrgyzstan' 'Iraq'
 'Vietnam' 'Myanmar' 'Oman' 'Philippines' 'Mauritania' 'Zimbabwe' 'Syria'
 'India' 'United Arab Emirates' 'Guatemala' 'Gabon' 'Honduras' 'Nicaragua'
 'Moldova' 'Sri Lanka' 'Bolivia' 'Algeria' 'Libya' 'Azerbaijan' 'Bahrain'
 'Namibia' 'Saudi Arabia' 'China' 'Botswana' 'Costa Rica' 'Qatar' 'Egypt'
 'Kuwait' 'Morocco' 'Paraguay' 'Ecuador' 'Iran' 'Armenia' 'El Salvador'
 'Turkmenistan' 'Mongolia' 'Mexico' 'Netherlands (Kingdom of the)' 'Japan'
 'Norway' 'Antigua and Barbuda' 'Czechia' 'Germany' 'Belgium' 'Ireland'
 'Denmark' 'Switzerland' 'Sweden' 'United States of America' 'Thailand'
 'Albania' 'Venezuela' 'Peru' 'Malaysia' 'South Africa' 'Panama' 'Jamaica'
 'Colombia' 'North Macedonia' 'Tunisia' 'Dominican Republic' 'Guyan

In [15]:
dirac[dirac.isna().any(axis=1)]


,Region Name,ISO3,Country_harmonized,Year_Units,RT_m,Year


In [16]:
dirac.info()


<class 'pandas.core.frame.DataFrame'>
Index: 468 entries, 10 to 480
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Region Name         468 non-null    object 
 1   ISO3                468 non-null    object 
 2   Country_harmonized  468 non-null    object 
 3   Year_Units          468 non-null    int64  
 4   RT_m                468 non-null    float64
 5   Year                468 non-null    int64  
dtypes: float64(1), int64(2), object(3)
memory usage: 25.6+ KB


##Variables encontradas
Entonces, con estos datos se tiene lo siguiente:

* 'Country_harmonized' = país al que corresponde la información

* 'Region Name' = región geográfica o continental en la que se localiza el país.

* 'RTCenters' = número de centros que disponen de al menos un equipo de radioterapia.

* 'Linacs' = número de centros con equipos de teleterapia que utilizan haces de fotones y electrones.

* 'Protontherapy'= número de centros con disponibilidad de protonterapia.

* 'XRay' = número de generadores de rayos X utilizados con fines terapéuticos.

* 'Brachytherapy' = número de equipos de braquiterapia, incluyendo tanto fuentes radiactivas como sistemas de braquiterapia electrónica mediante generadores miniaturizados de rayos X.

* 'Last Update' = año de la última actualización de la información registrada para cada país.

Las variables pueden clasificarse en cualitativas nominales, cuantitativas discretas y una variable temporal, lo que permite realizar posteriormente un análisis exploratorio orientado a la detección de valores atípicos (outliers) y a la normalización por población.

Ahora se realiza un análisis exploratorio para buscar outliers, si es que los hay.




In [17]:
categ_vars = dirac.select_dtypes(include='object').columns


In [18]:
#explotar las variables categoricas

for col in categ_vars:
    print(f"Valores únicos en {col}:")
    print(dirac[col].unique())
    print("-" * 40)

Valores únicos en Region Name:
['Africa' 'Eastern Mediterranean' 'Western Pacific' 'Europe'
 'South-East Asia' 'Americas']
----------------------------------------
Valores únicos en ISO3:
['MOZ' 'YEM' 'CMR' 'NGA' 'MLI' 'MDG' 'CIV' 'AGO' 'RWA' 'TZA' 'KHM' 'TJK'
 'BGD' 'GHA' 'LAO' 'SDN' 'UZB' 'ZMB' 'IDN' 'KEN' 'PAK' 'NPL' 'KGZ' 'IRQ'
 'VNM' 'MMR' 'OMN' 'PHL' 'MRT' 'ZWE' 'SYR' 'IND' 'ARE' 'GTM' 'GAB' 'HND'
 'NIC' 'MDA' 'LKA' 'BOL' 'DZA' 'LBY' 'AZE' 'BHR' 'NAM' 'SAU' 'CHN' 'BWA'
 'CRI' 'QAT' 'EGY' 'KWT' 'MAR' 'PRY' 'ECU' 'IRN' 'ARM' 'SLV' 'TKM' 'MNG'
 'MEX' 'NLD' 'JPN' 'NOR' 'ATG' 'CZE' 'DEU' 'BEL' 'IRL' 'DNK' 'CHE' 'SWE'
 'USA' 'THA' 'ALB' 'VEN' 'PER' 'MYS' 'ZAF' 'PAN' 'JAM' 'COL' 'MKD' 'TUN'
 'DOM' 'GUY' 'BRA' 'CUB' 'CHL' 'MUS' 'LBN' 'TUR' 'ROU' 'ARG' 'BIH' 'SRB'
 'KOR' 'KAZ' 'UKR' 'BRN' 'POL' 'RUS' 'ISR' 'BHS' 'SUR' 'SGP' 'GEO' 'HRV'
 'MCO' 'HUN' 'LVA' 'MNE' 'BLR' 'GRC' 'EST' 'SVK' 'GBR' 'URY' 'BRB' 'PRT'
 'NZL' 'BGR' 'TTO' 'SVN' 'ITA' 'CYP' 'AUT' 'ISL' 'MLT' 'FRA' 'ESP' 'AUS'
 'CAN' 'A

Con esto se demuestra que estos valores anómalos están en la última fila por consecuencia de sumar todos los equipos y centros y calcular la última actualización, entonces, esta fila se eliminará de los datos. Otra operación que sería interesante modificar son los nombres con caracteres que nos podría dar problemas, así como asegurarse de que no hay paises repetidos.

Debido a que hay países que están segregados por zonas, se unificarán usando Country_harmonized.

Durante el proceso de armonización geográfica se identificaron territorios no soberanos, como Martinica, que inicialmente se consideraron candidatos a ser integrados bajo el Estado correspondiente (Francia). No obstante, un análisis exploratorio de los datos epidemiológicos reveló diferencias sustanciales entre Francia y Martinica en las variables clave de GLOBOCAN, tales como la tasa estandarizada por edad (ASR World), la tasa bruta, el riesgo acumulado y el número total de casos. Por lo tanto, se usarán por separado, algo que no ocurre con UK, donde todo el territorio tiene resultados estadísticos similares, donde entonces si se sumarán casos y se promediarán los datos numeros como el ASR que son tasas.

Adicionalmente, se constató que ambas fuentes de datos GLOBOCAN y DIRAC, proporcionan información diferenciada para Martinica y Francia. En consecuencia, y con el fin de preservar la coherencia interna de los datos y evitar la introducción de sesgos derivados de agregaciones no justificadas, se decidió mantener Martinica como una entidad separada en el análisis.


In [19]:
# Comprobar duplicados por país y Year_Units
n_dups = dirac.duplicated(subset=["Country_harmonized", "Year_Units"]).sum()

print(f"Number of duplicates after cleaning: {n_dups}")


Number of duplicates after cleaning: 0


Dado que el único campo no numérico del conjunto DIRAC corresponde al año de la última actualización, no fue necesario aplicar reglas de agregación adicionales para variables categóricas.

In [20]:
dirac_clean = dirac.copy()


In [21]:
dirac_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 468 entries, 10 to 480
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Region Name         468 non-null    object 
 1   ISO3                468 non-null    object 
 2   Country_harmonized  468 non-null    object 
 3   Year_Units          468 non-null    int64  
 4   RT_m                468 non-null    float64
 5   Year                468 non-null    int64  
dtypes: float64(1), int64(2), object(3)
memory usage: 25.6+ KB


Se ha limpiado de caracteres anómalos en países.
Con los datos numéricos entonces, se analiza si existe algun dato anomalo

Con estos resultados, se puede saber que hay paises con valores extremos y otros donde escasamente hay un solo equipo. Se puede ver que hay paises donde es cero en equipos mientras que otros tienen hasta 2237 centros con radioterapia, esto tiene sentido si se explora que paises son así como su población, esto se hará más adelante con los datos del BM.


Con estos resultados es necesario tener entonces los datos demograficos por país, ademas de los datos de cancer.
Ahora, a explorar y depurar los datos provenientes de
#GLOBOCAN.
En donde se podrá encontrar los indice de incidencia y mortalidad segun sexo, tipo de cancer, pais y año.
Para esto, se ha tomado seis tipos de cancer muy comunes: Pulmon, mama, prostata, colon, cervix y leucemia.

In [22]:

df_all_cancer["Cancer label"].unique()


array(['Lung', 'Breast', 'Prostate', 'Cervix uteri'], dtype=object)

In [23]:
#se cambia solamente el nombre de Cervix uteri
df_all_cancer["Cancer label"] = df_all_cancer["Cancer label"].replace(
    {"Cervix uteri": "Cervix"}
)


In [24]:
df_all_cancer["Cancer label"].unique()

array(['Lung', 'Breast', 'Prostate', 'Cervix'], dtype=object)

In [25]:
df_all_cancer["Cancer label"].value_counts()

,count
Cancer label,
Lung,3280
Breast,1640
Prostate,1640
Cervix,1640


In [26]:
df_all_cancer.head()

,Cancer id,Cancer label,Population id,Country label,Sex,Type,Year,ASR (World),Crude rate,Cumulative risk,Total
0,11,Lung,112,Belarus,1,0,2007,61.434111,80.958928,7.865065,3666
1,11,Lung,112,Belarus,1,0,2008,62.729974,84.930604,8.033257,3765
2,11,Lung,112,Belarus,1,0,2009,61.098990,84.106229,7.731976,3719
3,11,Lung,112,Belarus,1,0,2010,62.582334,86.376743,7.914747,3812
4,11,Lung,112,Belarus,1,0,2011,59.755001,83.234410,7.621488,3665


Se puede observar que para un cáncer como el que afecta exclusivamente un sexo de la población , los datos son la mitad (1640 filas) del resto donde se toma en cuenta ambos sexos (3280 filas). En el caso de Estados Unidos, GLOBOCAN proporciona información adicional desagregada por grupos poblacionales (p. ej., “USA: White” y “USA: Black”), además de los datos correspondientes al total nacional. Dado que estas categorías no representan unidades geográficas independientes y que el conjunto de datos incluye la población total, dichas desagregaciones se excluyeron del análisis para evitar duplicidades y distorsiones en los indicadores epidemiológicos.

Los distintos conjuntos de datos correspondientes a cada tipo de cáncer se integraron en un único dataframe, incorporando una variable identificadora del tipo de cáncer para facilitar el análisis conjunto.


In [27]:
df_all_cancer = df_all_cancer.drop(columns=["Cancer id"], errors="ignore")




In [28]:

print(df_all_cancer.info())


for col in df_all_cancer:
    print(f"\n {col}")
    print(df_all_cancer[col].unique())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8200 entries, 0 to 8199
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Cancer label     8200 non-null   object 
 1   Population id    8200 non-null   int64  
 2   Country label    8200 non-null   object 
 3   Sex              8200 non-null   int64  
 4   Type             8200 non-null   int64  
 5   Year             8200 non-null   int64  
 6   ASR (World)      8200 non-null   float64
 7   Crude rate       8200 non-null   float64
 8   Cumulative risk  8200 non-null   float64
 9   Total            8200 non-null   int64  
dtypes: float64(3), int64(5), object(2)
memory usage: 640.8+ KB
None

 Cancer label
['Lung' 'Breast' 'Prostate' 'Cervix']

 Population id
[ 112  124  152  156  170  188  191  192  196  203  208  218  233  246
  250  268  276  300   32  320  328  348  352  356   36  372  376  380
  392   40  410  414  417  428  440  442  470  474   48  480 

Durante el análisis exploratorio se identificaron diferencias en la nomenclatura de países, incluyendo desagregaciones territoriales y subpoblacionales. Estas inconsistencias se abordaron mediante un proceso de armonización de nombres para permitir la integración con otras fuentes de datos. Es necesaior un nombre único por país.

In [29]:

#aplicando maping para cambiar y armonizar los nombres de paises
df_all_cancer["Country_harmonized"] = (
    df_all_cancer["Country label"]
    .replace(country_mapping)
)

# se normalizan los caracteres para la columna Country_harmonized, que ya existe.
df_all_cancer["Country_harmonized"] = df_all_cancer["Country_harmonized"].apply(
    lambda x: unicodedata.normalize("NFKD", x).encode("ASCII", "ignore").decode("utf-8")
    if pd.notna(x) else x
)


print(len(df_all_cancer["Country_harmonized"].unique()))

for col in df_all_cancer:
    print(f"\n {col}")
    print(df_all_cancer[col].unique())

# The 'Country label' column should no longer exist if the cell ran previously.
# If it somehow reappears, this line will ensure it is dropped without error.
df_all_cancer = df_all_cancer.drop(columns=["Country label"], errors="ignore")


76

 Cancer label
['Lung' 'Breast' 'Prostate' 'Cervix']

 Population id
[ 112  124  152  156  170  188  191  192  196  203  208  218  233  246
  250  268  276  300   32  320  328  348  352  356   36  372  376  380
  392   40  410  414  417  428  440  442  470  474   48  480  484  498
   51  528  554  558   56  578  591  600  608  616  620  630  634  642
  688  702  703  705  710  724  752  756   76  764  792  800  826 8260
 8261 8262 8263 8265   84  840 8401 8402  858  860  862]

 Country label
['Belarus' 'Canada' 'Chile' 'China' 'Colombia' 'Costa Rica' 'Croatia'
 'Cuba' 'Cyprus' 'Czechia' 'Denmark' 'Ecuador' 'Estonia' 'Finland'
 'France (metropolitan)' 'Georgia' 'Germany' 'Greece' 'Argentina'
 'Guatemala' 'Guyana' 'Hungary' 'Iceland' 'India' 'Australia' 'Ireland'
 'Israel' 'Italy' 'Japan' 'Austria' 'Korea, Republic of' 'Kuwait'
 'Kyrgyzstan' 'Latvia' 'Lithuania' 'Luxembourg' 'Malta'
 'France, Martinique' 'Bahrain' 'Mauritius' 'Mexico' 'Moldova' 'Armenia'
 'The Netherlands' 'New Zealan

In [30]:
df_all_cancer.columns


Index(['Cancer label', 'Population id', 'Sex', 'Type', 'Year', 'ASR (World)',
       'Crude rate', 'Cumulative risk', 'Total', 'Country_harmonized'],
      dtype='object')

In [31]:
type_map = {0: "Incidence", 1: "Mortality"}
sex_map  = {1: "Male", 2: "Female"}


df_all_cancer["Type"] = df_all_cancer["Type"].map(type_map)
df_all_cancer["Sex"]  = df_all_cancer["Sex"].map(sex_map)


df_all_cancer["Cancer label"].unique()

array(['Lung', 'Breast', 'Prostate', 'Cervix'], dtype=object)

In [32]:
key_cols = ["Country_harmonized", "Cancer label", "Sex", "Type", "Year"]
df_all_cancer.duplicated(subset=key_cols).sum()

np.int64(390)

In [33]:
df_all_cancer

,Cancer label,Population id,Sex,Type,Year,ASR (World),Crude rate,Cumulative risk,Total,Country_harmonized
0,Lung,112,Male,Incidence,2007,61.434111,80.958928,7.865065,3666,Belarus
1,Lung,112,Male,Incidence,2008,62.729974,84.930604,8.033257,3765,Belarus
2,Lung,112,Male,Incidence,2009,61.098990,84.106229,7.731976,3719,Belarus
3,Lung,112,Male,Incidence,2010,62.582334,86.376743,7.914747,3812,Belarus
4,Lung,112,Male,Incidence,2011,59.755001,83.234410,7.621488,3665,Belarus
...,...,...,...,...,...,...,...,...,...,...
8195,Cervix,862,Female,Mortality,2012,8.973664,8.901073,0.954095,1321,Venezuela
8196,Cervix,862,Female,Mortality,2013,9.133123,9.222048,0.970629,1386,Venezuela
8197,Cervix,862,Female,Mortality,2014,9.922533,10.276557,1.023536,1563,Venezuela
8198,Cervix,862,Female,Mortality,2015,9.740112,10.176412,1.023116,1565,Venezuela


Los datos oncológicos incluyen tanto incidencia como mortalidad, diferenciadas mediante la variable Type. Para los análisis comparativos entre países se emplean principalmente las tasas estandarizadas por edad (ASR (World)), al permitir comparaciones internacionales independientes de la estructura demográfica.
Las tasas de incidencia y mortalidad se expresan como ASR (World), calculadas mediante la estandarización directa usando la población estándar mundial.

Las variables que nos interesan por ahora, son:

* Cancer label: Nombre del tipo de cáncer analizado.

* Country_harmonized: País al que corresponden los datos.

* Sex: Sexo de la población considerada (1 hombre, 2 mujer).

* Year: Año de referencia del dato epidemiológico.

* Type: Tipo de medida epidemiológica, diferenciando entre incidencia y mortalidad (0 incidencia, 1 mortalidad).

* ASR (World): Tasa estandarizada por edad según la población estándar mundial, expresada por 100.000 habitantes.

* Crude rate: Tasa bruta de incidencia o mortalidad por 100.000 habitantes, no ajustada por edad.

* Cumulative risk: Riesgo acumulado de desarrollar o morir por el cáncer hasta una edad determinada.

* Total:número total de detecciones.

Los datos no relevantes para el estudio que son cancer id y population id, se eliminan.

En el caso del Reino Unido, GLOBOCAN proporciona información desagregada por subdivisiones territoriales. Dado que estas corresponden a una misma entidad nacional y que los valores de las tasas epidemiológicas eran similares entre las distintas subdivisiones, se procedió a integrar dichos registros. Los recuentos absolutos se agregaron mediante suma, mientras que las tasas (ASR World, tasa bruta y riesgo acumulado) se combinaron mediante media simple, dado que no se disponía de información poblacional para realizar una ponderación adecuada.

Entonces, a continuación realizaremos la implementación más adecuada posible, se separa UK del resto, se suma el **TOTAL** y se calcula la media del resto de valores.

In [34]:

uk = df_all_cancer[df_all_cancer["Country_harmonized"] == "United Kingdom"]
rest = df_all_cancer[df_all_cancer["Country_harmonized"] != "United Kingdom"]

uk_agg = (
    uk
    .groupby(
        ["Country_harmonized", "Cancer label", "Sex", "Type", "Year"],
        as_index=False
    )
    .agg({
        "Total": "sum",
        "ASR (World)": "mean",
        "Crude rate": "mean",
        "Cumulative risk": "mean"
    })
)

Con esto se tiene entonces UK unificado con medias en sus valores.
Una vez se ha explorado y corregido los datos de GLOBOCAN, se fusionan los datos que se tienen con lo obtenido con DIRAC y GLOBOCAN para hacer un solo dataset.

Adicionalmente, además de concatenar entonces UK co el resto, es necesario cambiar la momenclatura de sex y type para hacerlo más fácil de manipular e interpretar

In [35]:
# Ahora se reconstruye nuevamente

df_all_cancer_clean = pd.concat([rest, uk_agg], ignore_index=True)


#Con los datos ya limpios, se genera entonces un nuevo dataframe
glob_clean = df_all_cancer_clean.copy()

Para el
#Banco mundial
Se tiene entonces

In [36]:
#Se limpia el dataset del banco mundial

print(df_PIB[["country", "date", "value"]].head())

df_PIB["Country"] = df_PIB["country"].apply(lambda x: x["value"])
df_PIB["Year"] = df_PIB["date"].astype(int)
df_PIB["GDP_per_capita"] = df_PIB["value"]

print(df_PIB[["Country", "Year", "GDP_per_capita"]].head())
print(df_PIB["Country"].unique())


                                             country  date        value
0  {'id': 'ZH', 'value': 'Africa Eastern and Sout...  2024  1615.396356
1  {'id': 'ZH', 'value': 'Africa Eastern and Sout...  2023  1571.449189
2  {'id': 'ZH', 'value': 'Africa Eastern and Sout...  2022  1679.327622
3  {'id': 'ZH', 'value': 'Africa Eastern and Sout...  2021  1562.416175
4  {'id': 'ZH', 'value': 'Africa Eastern and Sout...  2020  1351.591669
                       Country  Year  GDP_per_capita
0  Africa Eastern and Southern  2024     1615.396356
1  Africa Eastern and Southern  2023     1571.449189
2  Africa Eastern and Southern  2022     1679.327622
3  Africa Eastern and Southern  2021     1562.416175
4  Africa Eastern and Southern  2020     1351.591669
['Africa Eastern and Southern' 'Africa Western and Central' 'Arab World'
 'Caribbean small states' 'Central Europe and the Baltics'
 'Early-demographic dividend' 'East Asia & Pacific'
 'East Asia & Pacific (excluding high income)'
 'East Asia & Pacif

In [37]:
df_PIB["Country_harmonized"] = (
    df_PIB["Country"]
    .replace(country_mapping)
)

df_PIB["Country_harmonized"] = df_PIB["Country_harmonized"].apply(
    lambda x: unicodedata.normalize("NFKD", x).encode("ASCII", "ignore").decode("utf-8")
    if pd.notna(x) else x
)
df_PIB["Country_harmonized"].unique()

array(['Africa Eastern and Southern', 'Africa Western and Central',
       'Arab World', 'Caribbean small states',
       'Central Europe and the Baltics', 'Early-demographic dividend',
       'East Asia & Pacific',
       'East Asia & Pacific (excluding high income)',
       'East Asia & Pacific (IDA & IBRD countries)', 'Euro area',
       'Europe & Central Asia',
       'Europe & Central Asia (excluding high income)',
       'Europe & Central Asia (IDA & IBRD countries)', 'European Union',
       'Fragile and conflict affected situations',
       'Heavily indebted poor countries (HIPC)', 'High income',
       'IBRD only', 'IDA & IBRD total', 'IDA blend', 'IDA only',
       'IDA total', 'Late-demographic dividend',
       'Latin America & Caribbean',
       'Latin America & Caribbean (excluding high income)',
       'Latin America & the Caribbean (IDA & IBRD countries)',
       'Least developed countries: UN classification',
       'Low & middle income', 'Low income', 'Lower middle in

In [38]:
df_PIB.head()

,countryiso3code,country,date,value,indicator,Country,Year,GDP_per_capita,Country_harmonized
0,AFE,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",2024,1615.396356,"{'id': 'NY.GDP.PCAP.CD', 'value': 'GDP per cap...",Africa Eastern and Southern,2024,1615.396356,Africa Eastern and Southern
1,AFE,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",2023,1571.449189,"{'id': 'NY.GDP.PCAP.CD', 'value': 'GDP per cap...",Africa Eastern and Southern,2023,1571.449189,Africa Eastern and Southern
2,AFE,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",2022,1679.327622,"{'id': 'NY.GDP.PCAP.CD', 'value': 'GDP per cap...",Africa Eastern and Southern,2022,1679.327622,Africa Eastern and Southern
3,AFE,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",2021,1562.416175,"{'id': 'NY.GDP.PCAP.CD', 'value': 'GDP per cap...",Africa Eastern and Southern,2021,1562.416175,Africa Eastern and Southern
4,AFE,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",2020,1351.591669,"{'id': 'NY.GDP.PCAP.CD', 'value': 'GDP per cap...",Africa Eastern and Southern,2020,1351.591669,Africa Eastern and Southern


In [39]:
df_PIB = df_PIB.drop(columns=["country", "date", "value", "Country", "indicator"], errors="ignore")
datos_numericos = df_PIB.select_dtypes(include=['int64', "float64"])

datos_numericos.describe().T

,count,mean,std,min,25%,50%,75%,max
Year,17290.0,1992.000000,18.762206,1960.000000,1976.000000,1992.000000,2008.000000,2024.000000
GDP_per_capita,14561.0,8701.887816,17595.339722,11.801322,583.782776,1938.810849,8045.946305,288001.433369


In [40]:
df_PIB["Country_harmonized"].unique()

array(['Africa Eastern and Southern', 'Africa Western and Central',
       'Arab World', 'Caribbean small states',
       'Central Europe and the Baltics', 'Early-demographic dividend',
       'East Asia & Pacific',
       'East Asia & Pacific (excluding high income)',
       'East Asia & Pacific (IDA & IBRD countries)', 'Euro area',
       'Europe & Central Asia',
       'Europe & Central Asia (excluding high income)',
       'Europe & Central Asia (IDA & IBRD countries)', 'European Union',
       'Fragile and conflict affected situations',
       'Heavily indebted poor countries (HIPC)', 'High income',
       'IBRD only', 'IDA & IBRD total', 'IDA blend', 'IDA only',
       'IDA total', 'Late-demographic dividend',
       'Latin America & Caribbean',
       'Latin America & Caribbean (excluding high income)',
       'Latin America & the Caribbean (IDA & IBRD countries)',
       'Least developed countries: UN classification',
       'Low & middle income', 'Low income', 'Lower middle in

In [41]:
print(df_PIB.isna().sum().sort_values(ascending=False))
(df_PIB.isna().mean() * 100).sort_values(ascending=False)




GDP_per_capita        2729
countryiso3code          0
Year                     0
Country_harmonized       0
dtype: int64


,0
GDP_per_capita,15.78369
countryiso3code,0.00000
Year,0.00000
Country_harmonized,0.00000


In [42]:

df_PIB_clean = df_PIB.dropna(subset=["GDP_per_capita"])

df_PIB_clean.isna().sum()

,0
countryiso3code,0
Year,0
GDP_per_capita,0
Country_harmonized,0


Para los equipos de radiodiagnostico

In [43]:
diagnostico.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 661 entries, 0 to 660
Data columns (total 13 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   CountryCode                       661 non-null    object 
 1   Location                          661 non-null    object 
 2   RegionCode                        661 non-null    object 
 3   Region                            661 non-null    object 
 4   Period                            661 non-null    object 
 5   Computed tomography units         322 non-null    float64
 6   Gamma camera or Nuclear medicine  277 non-null    float64
 7   Linear Accelerator                331 non-null    float64
 8   Magnetic Resonance Imaging        315 non-null    float64
 9   Mammography units                 291 non-null    float64
 10  Positron Emission tomography      274 non-null    float64
 11  Radiotherapy units                481 non-null    float64
 12  Telecoba

In [44]:
diagnostico.head(10)

,CountryCode,Location,RegionCode,Region,Period,Computed tomography units,Gamma camera or Nuclear medicine,Linear Accelerator,Magnetic Resonance Imaging,Mammography units,Positron Emission tomography,Radiotherapy units,Telecobalt Unit
0,AFG,Afghanistan,EMR,Eastern Mediterranean,2010,0.19,0.0,0.0,0.100,0.00,0.0,0.000,0.000
1,AFG,Afghanistan,EMR,Eastern Mediterranean,2013,0.20,0.0,0.0,0.098,NaN,0.0,0.000,0.000
2,AFG,Afghanistan,EMR,Eastern Mediterranean,2014,NaN,NaN,NaN,NaN,0.00,NaN,NaN,NaN
3,AGO,Angola,AFR,Africa,2010,0.47,0.0,0.0,0.050,6.98,0.0,0.050,0.050
4,AGO,Angola,AFR,Africa,2013,0.42,0.0,0.0,0.047,NaN,0.0,0.047,0.047
5,AGO,Angola,AFR,Africa,2014,NaN,NaN,NaN,NaN,6.33,NaN,NaN,NaN
6,AGO,Angola,AFR,Africa,2017-2021,NaN,NaN,NaN,NaN,NaN,NaN,0.120,NaN
7,ALB,Albania,EUR,Europe,2010,5.31,0.0,0.0,1.560,62.71,0.0,0.310,0.310
8,ALB,Albania,EUR,Europe,2013,5.36,0.0,0.0,1.580,NaN,0.0,0.320,0.320
9,ALB,Albania,EUR,Europe,2014,NaN,NaN,NaN,NaN,54.40,NaN,NaN,NaN


In [45]:
print(diagnostico.head())
print(diagnostico["Period"].unique())


#cuantas filas han quedado con el tipo de cancer, sexo, año y pais.
diagnostico.shape

  CountryCode     Location RegionCode                 Region Period  \
0         AFG  Afghanistan        EMR  Eastern Mediterranean   2010   
1         AFG  Afghanistan        EMR  Eastern Mediterranean   2013   
2         AFG  Afghanistan        EMR  Eastern Mediterranean   2014   
3         AGO       Angola        AFR                 Africa   2010   
4         AGO       Angola        AFR                 Africa   2013   

   Computed tomography units  Gamma camera or Nuclear medicine  \
0                       0.19                               0.0   
1                       0.20                               0.0   
2                        NaN                               NaN   
3                       0.47                               0.0   
4                       0.42                               0.0   

   Linear Accelerator  Magnetic Resonance Imaging  Mammography units  \
0                 0.0                       0.100               0.00   
1                 0.0           

(661, 13)

In [46]:
# List the columns you want to drop
columns_to_drop = [
    "Radiotherapy units",
    "Telecobalt Unit",
    "RegionCode",
    "Region",
    "Linear Accelerator"
]

# Drop the columns from the DataFrame (e.g., dirac)
# I will use 'dirac' as it was the last dataframe being processed before the current context.
# If you want to drop columns from a different dataframe, replace 'dirac' with its name.
# The 'errors="ignore"' argument prevents an error if a column in the list doesn't exist.
diagnostico = diagnostico.drop(columns=columns_to_drop, errors="ignore")
diagnostico.head(5)

,CountryCode,Location,Period,Computed tomography units,Gamma camera or Nuclear medicine,Magnetic Resonance Imaging,Mammography units,Positron Emission tomography
0,AFG,Afghanistan,2010,0.19,0.0,0.100,0.00,0.0
1,AFG,Afghanistan,2013,0.20,0.0,0.098,NaN,0.0
2,AFG,Afghanistan,2014,NaN,NaN,NaN,0.00,NaN
3,AGO,Angola,2010,0.47,0.0,0.050,6.98,0.0
4,AGO,Angola,2013,0.42,0.0,0.047,NaN,0.0


In [47]:
nan_count = diagnostico.isna().sum()
nan_count


,0
CountryCode,0
Location,0
Period,0
Computed tomography units,339
Gamma camera or Nuclear medicine,384
Magnetic Resonance Imaging,346
Mammography units,370
Positron Emission tomography,387


In [48]:
diagnostico.shape

(661, 8)

In [49]:
#diagnostico = diagnostico.fillna(0)

diagnostico.info()
print(diagnostico.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 661 entries, 0 to 660
Data columns (total 8 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   CountryCode                       661 non-null    object 
 1   Location                          661 non-null    object 
 2   Period                            661 non-null    object 
 3   Computed tomography units         322 non-null    float64
 4   Gamma camera or Nuclear medicine  277 non-null    float64
 5   Magnetic Resonance Imaging        315 non-null    float64
 6   Mammography units                 291 non-null    float64
 7   Positron Emission tomography      274 non-null    float64
dtypes: float64(5), object(3)
memory usage: 41.4+ KB
CountryCode                           0
Location                              0
Period                                0
Computed tomography units           339
Gamma camera or Nuclear medicine    384
Magnetic Reso

In [50]:
diagnostico.columns = diagnostico.columns.str.strip()

diagnostico = diagnostico[[
    'Period',
    "Computed tomography units",
    "Gamma camera or Nuclear medicine",
    "Magnetic Resonance Imaging",
    "Mammography units",
    "Positron Emission tomography",
    "CountryCode",
]].rename(columns={
    "Period": "Year",
    "Computed tomography units": "TAC",
    "Gamma camera or Nuclear medicine": "GammaC",
    "Magnetic Resonance Imaging": "MRI",
    "Mammography units": "Mammography",
    "Positron Emission tomography": "PET",
    "CountryCode" : "ISO3"
})

diagnostico["Year"] = (
    diagnostico["Year"]
    .astype(str)
    .str.strip()
    .str.extract(r"(\d{4})", expand=False)
)

diagnostico["Year"] = pd.to_numeric(diagnostico["Year"], errors="coerce")

diagnostico['Year'] = diagnostico['Year'].replace('2017-2021', '2017')
diagnostico['Year'] = diagnostico['Year'].replace('2019-2021', '2019')
diagnostico['Year'] = diagnostico['Year'].astype(int)
diagnostico["Year"] = diagnostico["Year"] + 5

diagnostico.head()

,Year,TAC,GammaC,MRI,Mammography,PET,ISO3
0,2015,0.19,0.0,0.100,0.00,0.0,AFG
1,2018,0.20,0.0,0.098,NaN,0.0,AFG
2,2019,NaN,NaN,NaN,0.00,NaN,AFG
3,2015,0.47,0.0,0.050,6.98,0.0,AGO
4,2018,0.42,0.0,0.047,NaN,0.0,AGO


In [51]:
diagnostico.to_csv(output_path / "diagnostico_clean.csv", index=False)

To check for NaN values in a specific column, you can use the following code. Replace `"Your_Column_Name"` with the actual name of the column you want to check.

In [52]:
glob_clean.head()

,Cancer label,Population id,Sex,Type,Year,ASR (World),Crude rate,Cumulative risk,Total,Country_harmonized
0,Lung,112.0,Male,Incidence,2007,61.434111,80.958928,7.865065,3666,Belarus
1,Lung,112.0,Male,Incidence,2008,62.729974,84.930604,8.033257,3765,Belarus
2,Lung,112.0,Male,Incidence,2009,61.098990,84.106229,7.731976,3719,Belarus
3,Lung,112.0,Male,Incidence,2010,62.582334,86.376743,7.914747,3812,Belarus
4,Lung,112.0,Male,Incidence,2011,59.755001,83.234410,7.621488,3665,Belarus


In [53]:
#Unimos population de ONU con globocan
ONU_c_merged = ONU_c.merge(glob_clean,
                           on=["Population id", "Year", "Country_harmonized"],
                           how="inner"
)



In [54]:
ONU_c_merged.head()

,Country_harmonized,Year,Population,Population_male,Population_female,ISO3,Population id,Cancer label,Sex,Type,ASR (World),Crude rate,Cumulative risk,Total
0,Mauritius,2007,1269518.0,638687.0,630832.0,MUS,480,Lung,Male,Mortality,14.055004,12.682278,1.702089,81
1,Mauritius,2007,1269518.0,638687.0,630832.0,MUS,480,Lung,Female,Mortality,4.489244,5.231187,0.480463,33
2,Mauritius,2007,1269518.0,638687.0,630832.0,MUS,480,Breast,Female,Mortality,12.746928,14.425394,1.416181,91
3,Mauritius,2007,1269518.0,638687.0,630832.0,MUS,480,Prostate,Male,Mortality,7.897107,6.889139,0.899300,44
4,Mauritius,2007,1269518.0,638687.0,630832.0,MUS,480,Cervix,Female,Mortality,2.693513,2.853375,0.394047,18


Una vez se tienen los datos limpios, se busca paises en común.

In [55]:
df_PIB_clean.head()

,countryiso3code,Year,GDP_per_capita,Country_harmonized
0,AFE,2024,1615.396356,Africa Eastern and Southern
1,AFE,2023,1571.449189,Africa Eastern and Southern
2,AFE,2022,1679.327622,Africa Eastern and Southern
3,AFE,2021,1562.416175,Africa Eastern and Southern
4,AFE,2020,1351.591669,Africa Eastern and Southern


In [56]:
#glob_c = set(glob_clean["Country_harmonized"].dropna().unique())
dir_c  = set(dirac_clean["ISO3"].dropna().unique())
diag_c = set(diagnostico["ISO3"].dropna().unique())
pop_c  = set(ONU_c_merged["ISO3"].dropna().unique())
BM_c  = set(df_PIB_clean["countryiso3code"].dropna().unique())

common_4 = sorted(dir_c & pop_c & BM_c & diag_c)
print("Número de países en común:", len(common_4))



Número de países en común: 70


In [57]:
def clean_country_set(s):
    return {
        c for c in s
        if ":" not in c
    }

#glob_countries_clean = clean_country_set(glob_c)
dirac_countries_clean = clean_country_set(dir_c)
onu_countries_clean = clean_country_set(pop_c)
BM_countries_clean = clean_country_set(BM_c)
diag_countries_clean = clean_country_set(diag_c)

common_4_clean = (
    diag_countries_clean
    & dirac_countries_clean
    & onu_countries_clean
    & BM_countries_clean
)

print(len(common_4_clean))
for c in sorted(common_4_clean):
    print(c)



70
ARG
ARM
AUS
AUT
BEL
BHR
BLR
BLZ
BRA
CAN
CHE
CHL
CHN
COL
CRI
CUB
CYP
CZE
DEU
DNK
ECU
ESP
EST
FIN
FRA
GEO
GRC
GTM
GUY
HRV
HUN
IND
IRL
ISL
ISR
ITA
JPN
KGZ
KOR
KWT
LTU
LUX
LVA
MDA
MEX
MLT
MUS
NIC
NOR
NZL
PAN
PHL
POL
PRT
PRY
QAT
ROU
SGP
SRB
SVK
SVN
SWE
THA
TUR
UGA
URY
USA
UZB
VEN
ZAF


Tras el proceso de armonización de países, se identificaron 71 países con información disponible simultáneamente en los conjuntos de datos de GLOBOCAN, ONU, BM y DIRAC.
Ahora, se procede con la fusión de los datos por país y año, empezando con un INNER JOIN y comprobando los números de países disponibles.


In [58]:
ONU_c_merged.drop(columns=["Country_harmonized"], inplace=True)


In [59]:
ONU_c_merged.head()

,Year,Population,Population_male,Population_female,ISO3,Population id,Cancer label,Sex,Type,ASR (World),Crude rate,Cumulative risk,Total
0,2007,1269518.0,638687.0,630832.0,MUS,480,Lung,Male,Mortality,14.055004,12.682278,1.702089,81
1,2007,1269518.0,638687.0,630832.0,MUS,480,Lung,Female,Mortality,4.489244,5.231187,0.480463,33
2,2007,1269518.0,638687.0,630832.0,MUS,480,Breast,Female,Mortality,12.746928,14.425394,1.416181,91
3,2007,1269518.0,638687.0,630832.0,MUS,480,Prostate,Male,Mortality,7.897107,6.889139,0.899300,44
4,2007,1269518.0,638687.0,630832.0,MUS,480,Cervix,Female,Mortality,2.693513,2.853375,0.394047,18


In [60]:
df_PIB_clean.head()

,countryiso3code,Year,GDP_per_capita,Country_harmonized
0,AFE,2024,1615.396356,Africa Eastern and Southern
1,AFE,2023,1571.449189,Africa Eastern and Southern
2,AFE,2022,1679.327622,Africa Eastern and Southern
3,AFE,2021,1562.416175,Africa Eastern and Southern
4,AFE,2020,1351.591669,Africa Eastern and Southern


In [61]:
dirac_clean.drop(columns=["Region Name", "Country_harmonized"], errors="ignore"),

(    ISO3  Year_Units  RT_m  Year
 10   MOZ        2017  0.03  2022
 11   YEM        2017  0.03  2022
 12   CMR        2017  0.04  2022
 13   NGA        2017  0.04  2022
 14   MLI        2017  0.05  2022
 ..   ...         ...   ...   ...
 476  SWE        2010  8.32  2015
 477  CAN        2010  8.73  2015
 478  BEL        2010  8.96  2015
 479  DNK        2010  9.73  2015
 480  CHE        2010  9.79  2015
 
 [468 rows x 4 columns],)

In [62]:
df_merged = ONU_c_merged.merge(
    dirac_clean,
    on=["ISO3", "Year"],
    how="inner"
).merge(
    df_PIB_clean,
    left_on=["ISO3", "Year"],
    right_on=["countryiso3code", "Year"],
    how="inner"
).merge(
    diagnostico,
    on=["ISO3", "Year"],
    how="inner"
)






In [63]:
print(df_merged['ISO3'].unique())
print(len(df_merged['ISO3'].unique()))
print(df_merged["Year"].unique())
print(df_merged.head())


#se separan por sexo

df_male = df_merged[df_merged["Sex"] == "Male"].copy()
df_female = df_merged[df_merged["Sex"] == "Female"].copy()

['MUS' 'UGA' 'ZAF' 'KGZ' 'UZB' 'CHN' 'JPN' 'KOR' 'IND' 'PHL' 'SGP' 'THA'
 'ARM' 'CYP' 'GEO' 'ISR' 'KWT' 'QAT' 'TUR' 'BLR' 'CZE' 'HUN' 'POL' 'MDA'
 'ROU' 'SVK' 'DNK' 'EST' 'FIN' 'ISL' 'IRL' 'LVA' 'LTU' 'NOR' 'SWE' 'HRV'
 'GRC' 'ITA' 'MLT' 'PRT' 'SRB' 'SVN' 'ESP' 'AUT' 'BEL' 'FRA' 'DEU' 'LUX'
 'CHE' 'CUB' 'BLZ' 'CRI' 'GTM' 'MEX' 'NIC' 'PAN' 'ARG' 'BRA' 'CHL' 'COL'
 'ECU' 'GUY' 'PRY' 'URY' 'VEN' 'CAN' 'USA' 'AUS' 'NZL']
69
[2015 2018 2022]
   Year  Population  Population_male  Population_female ISO3  Population id  \
0  2015   1292275.0         648341.0           643934.0  MUS            480   
1  2015   1292275.0         648341.0           643934.0  MUS            480   
2  2015   1292275.0         648341.0           643934.0  MUS            480   
3  2015   1292275.0         648341.0           643934.0  MUS            480   
4  2015   1292275.0         648341.0           643934.0  MUS            480   

  Cancer label     Sex       Type  ASR (World)  ...  Year_Units  RT_m  \
0         L

In [64]:
#Comprobando que la fusión se ha realizado con éxito con los 71 paises.




print(df_merged["Year"].unique())

print("Numero de duplicados:", df_merged.duplicated(
    subset=["Cancer label", "Sex", "Type", "Year", "Year_Units"]
).sum())

#cuantas filas han quedado con el tipo de cancer, sexo, año y pais.
df_merged.head()

[2015 2018 2022]
Numero de duplicados: 1055


,Year,Population,Population_male,Population_female,ISO3,Population id,Cancer label,Sex,Type,ASR (World),...,Year_Units,RT_m,countryiso3code,GDP_per_capita,Country_harmonized_y,TAC,GammaC,MRI,Mammography,PET
0,2015,1292275.0,648341.0,643934.0,MUS,480,Lung,Male,Mortality,14.084766,...,2010,2.31,MUS,9630.543784,Mauritius,6.16,2.31,4.62,57.98,0.0
1,2015,1292275.0,648341.0,643934.0,MUS,480,Lung,Female,Mortality,5.710436,...,2010,2.31,MUS,9630.543784,Mauritius,6.16,2.31,4.62,57.98,0.0
2,2015,1292275.0,648341.0,643934.0,MUS,480,Breast,Female,Mortality,19.737396,...,2010,2.31,MUS,9630.543784,Mauritius,6.16,2.31,4.62,57.98,0.0
3,2015,1292275.0,648341.0,643934.0,MUS,480,Prostate,Male,Mortality,8.993800,...,2010,2.31,MUS,9630.543784,Mauritius,6.16,2.31,4.62,57.98,0.0
4,2015,1292275.0,648341.0,643934.0,MUS,480,Cervix,Female,Mortality,3.731733,...,2010,2.31,MUS,9630.543784,Mauritius,6.16,2.31,4.62,57.98,0.0


In [65]:
df_merged.drop(columns=["countryiso3code", "Country_harmonized_y", "Country_harmonized_x"], inplace=True)

In [66]:
#cuantos NaN
df_merged.isna().sum()[df_merged.isna().sum() > 0]



,0
TAC,445
GammaC,525
MRI,435
Mammography,715
PET,485


In [67]:
df_merged.shape

(1085, 22)

Teniendo entre 40% y más del 60% de estas variables con missing, no se utilizarán como contínuas para el modelo que se utilizará más adelante ya que es un missing estructural masivo, por lo tanto, se utilizarán como descriptivas, es importante justificar que se ha llegado hasta aquí con estas variables sin eliminar o cambiar porque se sospechaba que tras la fusión, muchos de esos missing no estarían en el dataset, sin embargo, no fue así, durante el EDA se continuará.

Años a estudiar los efectos a 5 años de los equipos [2015 2018 2022 2019]

In [68]:
dups_2022 = (
    df_all_cancer[df_all_cancer["Year"] == 2022]
    .loc[lambda x: x.duplicated(subset=key_cols, keep=False)]
    .sort_values(key_cols)
)

dups_2022.head(20)

,Cancer label,Population id,Sex,Type,Year,ASR (World),Crude rate,Cumulative risk,Total,Country_harmonized
4770,Breast,8263,Female,Mortality,2022,12.546886,30.805814,1.293370,299,United Kingdom
4786,Breast,8265,Female,Mortality,2022,13.244247,32.333348,1.353525,9934,United Kingdom
8050,Cervix,8263,Female,Mortality,2022,1.391916,2.163619,0.135046,21,United Kingdom
8066,Cervix,8265,Female,Mortality,2022,1.466227,2.555031,0.153706,785,United Kingdom
2981,Lung,8263,Female,Mortality,2022,19.447979,49.042032,2.314522,476,United Kingdom
3014,Lung,8265,Female,Mortality,2022,16.094501,44.558440,1.896991,13690,United Kingdom
2954,Lung,8263,Male,Mortality,2022,24.735880,57.024492,2.752872,536,United Kingdom
2997,Lung,8265,Male,Mortality,2022,20.164009,50.265025,2.280899,14839,United Kingdom
6410,Prostate,8263,Male,Mortality,2022,10.615046,29.788914,0.801072,280,United Kingdom
6426,Prostate,8265,Male,Mortality,2022,11.702751,37.271115,0.852256,11003,United Kingdom


In [69]:
# se guarda el dataset depurado
df_merged.to_csv(output_path / "df_merged_WHO_diag_.csv", index=False)
#df_merged_WHO es entonces según lo encontrado con la WHO, onde indica densidad
#de equipos de RT según el año

Los datos se encuentran limpios y fucionados en un solo dataframe. Ahora se realiza una exploración previa.

Antes de abordar los análisis de relación, se realizó un análisis exploratorio visual estratificado por año y sexo, con el objetivo de contextualizar la evolución temporal de la incidencia y la mortalidad por cáncer. Este análisis permitió identificar el periodo más adecuado para el estudio principal, teniendo en cuenta posibles distorsiones en los indicadores de salud asociadas a acontecimientos excepcionales, como la pandemia de COVID-19 en el 2020, que alteró significativamente los patrones de diagnóstico y tratamiento. Con esto además, podemos visualizar como evoluciona la incidencia y mortalidad en el timepo, si existen diferencias sistemáticas por sexo y detectar patrones globales.

